In [1]:
# Loading libraries
import requests
import re
from bs4 import BeautifulSoup
from time import sleep
import pandas as pd
from tqdm import tqdm

# Getting the html from our desired URL as a text string
url = 'https://carpentries.org/workshops/upcoming-workshops/'
req = requests.get(url).text

In [2]:
# Cleaning and printing the string
cleaned_req = re.sub(r'\s*\n\s*','', req).strip()
print(cleaned_req[0:1000])

<!doctype html><html class=scroll-smooth lang=en-us dir=ltr><head><meta charset=utf-8><meta name=viewport content="width=device-width"><title>Upcoming workshops | The Carpentries</title><link rel=preconnect href=https://fonts.googleapis.com><link rel=preconnect href=https://fonts.gstatic.com crossorigin><link href="https://fonts.googleapis.com/css2?family=Mulish:ital,wght@0,200..1000;1,200..1000&display=swap" rel=stylesheet><script defer src=https://cdn.jsdelivr.net/npm/@glidejs/glide@3.5.x></script><script src=https://kit.fontawesome.com/3a6fac633d.js crossorigin=anonymous></script><link rel=stylesheet href=https://cdn.datatables.net/1.13.6/css/jquery.dataTables.min.css><script src=https://code.jquery.com/jquery-3.7.1.min.js></script><script src=https://cdn.datatables.net/1.13.6/js/jquery.dataTables.min.js></script><script src=https://cdn.jsdelivr.net/npm/moment@2.29.1/moment.min.js></script><script src=https://cdn.datatables.net/plug-ins/1.13.6/sorting/datetime-moment.js></script><sc

In [3]:
# Parsing the HTML with BeautifulSoup
soup = BeautifulSoup(cleaned_req, 'html.parser')

# Finding all third-level headers and doing a formatted print
h3_by_tag = soup.find_all('h3')
print("Number of h3 elements found: ", len(h3_by_tag))
for n, h3 in enumerate(h3_by_tag):
    print(f"Workshop #{n} - {h3.get_text()}")

Number of h3 elements found:  9
Workshop #0 - Charles R. Drew University of Medicine and Science
Workshop #1 - University of New Mexico
Workshop #2 - University College London
Workshop #3 - University of Otago
Workshop #4 - Aarhus University
Workshop #5 - University of California Santa Barbara
Workshop #6 - The Ohio State University
Workshop #7 - Verein Deutscher Bibliothekarinnen und Bibliothekare e.V.
Workshop #8 - Digital Competence Center Praktijkgericht Onderzoek (DCC-PO)


In [4]:
# An alternative using the # An alternative using the "class" attribute, instead of the h3 tag
h3_by_class = soup.find_all(class_="title text-base md:text-[1.75rem] leading-[2.125rem] font-semibold")


In [5]:
# Get the parent of the first h3 element and prettify it
firsth3_parent = h3_by_class[0].parent
#print(str(div_firsth3))
print(firsth3_parent.prettify())

<div class="p-8 mb-5 border" data-country="United States" data-curriculum="Mix &amp; Match" data-meeting="Online" data-program="The Carpentries">
 <div class="flex mb-4 -mx-2">
  <div class="flex items-center mx-2">
   <img alt="" class="mx-1" src="/carpentries.svg"/>
   <span class="text-[0.625rem] uppercase">
    The Carpentries
   </span>
  </div>
  <div class="flex items-center mx-2">
   <img alt="" class="mr-1" height="20" src="/flags/us.png" width="20"/>
   <span class="text-[0.625rem] uppercase">
    United States
   </span>
  </div>
  <div class="flex items-center mx-2">
   <img alt="" class="mx-1" src="/Online.svg"/>
   <span class="text-[0.625rem] uppercase">
    Online
   </span>
  </div>
 </div>
 <h3 class="title text-base md:text-[1.75rem] leading-[2.125rem] font-semibold">
  <a class="underline hover:text-blue-hover text-gray-dark" href="https://meghalgandhi1019.github.io/2026-07-28-cdu-online/">
   Charles R. Drew University of Medicine and Science
  </a>
 </h3>
 <div cl

In [6]:
dict_workshop = {}
dict_workshop['host'] = firsth3_parent.find('h3').get_text()
dict_workshop['link'] = firsth3_parent.find('h3').find('a').get('href')
dict_workshop['curriculum'] = firsth3_parent.get('data-curriculum')
dict_workshop['country'] = firsth3_parent.get('data-country')
dict_workshop['format'] = firsth3_parent.get('data-meeting')
dict_workshop['program'] = firsth3_parent.get('data-program')

In [7]:
# Find all divs that match a class attribute
divs = soup.find_all('div', class_="p-8 mb-5 border")

workshop_list = []
for item in divs:
    dict_workshop = {}
    dict_workshop['host'] = item.find('h3').get_text()
    dict_workshop['link'] = item.find('h3').find('a').get('href')
    dict_workshop['curriculum'] = item.get('data-curriculum')
    dict_workshop['country'] = item.get('data-country')
    dict_workshop['format'] = item.get('data-meeting')
    dict_workshop['program'] = item.get('data-program')
    workshop_list.append(dict_workshop)

upcoming_workshops_df = pd.DataFrame(workshop_list)

In [8]:
# Get HTML and parse it with BeautifulSoup
url_past = 'https://carpentries.org/workshops/past-workshops/'
req_past = requests.get(url_past).text

soup_past = BeautifulSoup(req_past, 'html.parser')

# Find all divs that match a class attribute
divs_past = soup_past.find_all('div', class_="p-8 mb-5 border")

# Create an empty list, and fill it with info on each of the workshops found
workshop_list = []
for item in divs_past:
    dict_workshop = {}
    dict_workshop['host'] = item.find('h3').get_text()
    dict_workshop['link'] = item.find('h3').find('a').get('href')
    dict_workshop['curriculum'] = item.get('data-curriculum')
    dict_workshop['country'] = item.get('data-country')
    dict_workshop['format'] = item.get('data-meeting')
    dict_workshop['program'] = item.get('data-program')
    workshop_list.append(dict_workshop)

# Transform list into a DataFrame
pastworkshops_df  = pd.DataFrame(workshop_list)

print('Total number of workshops in the table: ', len(pastworkshops_df))

print('Top 5 of countries by number of workshops held: \n',
      pastworkshops_df['country'].value_counts().head())

Total number of workshops in the table:  4186
Top 5 of countries by number of workshops held: 
 country
United States     1961
United Kingdom     552
Australia          349
Canada             232
Germany            211
Name: count, dtype: int64


In [9]:
from time import sleep
print('First')
sleep(5)
print('Second')

First
Second


In [10]:
first_url = upcoming_workshops_df.loc[0, 'link']
print("URL we are visiting: ", first_url)

req = requests.get(first_url).text
cleaned_req = re.sub(r'\s*\n\s*', '', req).strip()

soup = BeautifulSoup(cleaned_req, 'html.parser')

URL we are visiting:  https://meghalgandhi1019.github.io/2026-07-28-cdu-online/


In [13]:
urls = list(upcoming_workshops_df.loc[:5, 'link'])
urls

['https://meghalgandhi1019.github.io/2026-07-28-cdu-online/',
 'https://jonathanwheeler01.github.io/2026-07-28-unm/',
 'https://github-pages.arc.ucl.ac.uk/2026-07-29-UCL/',
 'https://otagocarpentries.github.io/2026-07-30-nz-unix/',
 'https://jdlanz.github.io/2026-07-31-Aarhus/',
 'https://emlab-ucsb.github.io/2026-08-04-ucsb-intro-geospatial/']

In [12]:
list_of_workshops = []

for item in tqdm(urls):
    req = requests.get(item).text
    cleaned_req = re.sub(r'\s*\n\s*', '', req).strip()
    soup = BeautifulSoup(cleaned_req, 'html.parser')

    dict_w = {}
    dict_w['link'] = item

    dict_w['startdate'] = soup.find('meta', attrs = {'name': 'startdate'}).get('content')
    dict_w['enddate'] = soup.find('meta', attrs = {'name': 'enddate'}).get('content')
    dict_w['language'] = soup.find('meta', attrs = {'name': 'language'}).get('content')
    dict_w['latlng'] = soup.find('meta', attrs = {'name': 'latlng'}).get('content')
    dict_w['instructor'] = soup.find('meta', attrs = {'name': 'instructor'}).get('content')
    dict_w['helper'] = soup.find('meta', attrs = {'name': 'helper'}).get('content')

    list_of_workshops.append(dict_w)

    sleep(3)

extradata_upcoming_df = pd.DataFrame(list_of_workshops)
    

 83%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▏                            | 5/6 [00:17<00:03,  3.41s/it]


AttributeError: 'NoneType' object has no attribute 'get'

In [14]:
response = requests.get(first_url)
status_code = response.status_code
print(status_code)

200


In [15]:
if status_code == 200:
    # proceed with scraping
else:
    # handle or skip this URL

IndentationError: expected an indented block after 'if' statement on line 1 (12219115.py, line 3)